# LungScan AI — Baseline Model Notebook

Notebook permulaan untuk training model CT-scan paru 4 kelas:

- adenocarcinoma
- large_cell_carcinoma
- squamous_cell_carcinoma
- normal

Karena dua dataset Kaggle yang dicek ternyata sama/mirror, notebook ini default memakai **1 dataset utama saja** dan menyediakan deteksi duplikat dengan SHA256 agar tidak terjadi data leakage.

## 0. Setup
Jalankan di Google Colab atau Kaggle Notebook. Jika di Colab, upload `kaggle.json`.

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

!pip -q install kaggle scikit-learn pandas matplotlib pillow tqdm

## 1. Kaggle API
Lewati cell upload jika berjalan di Kaggle Notebook.

In [ ]:
from pathlib import Path

IS_COLAB = False
try:
    import google.colab
    IS_COLAB = True
except Exception:
    IS_COLAB = False

if IS_COLAB:
    from google.colab import files
    if not Path("/root/.kaggle/kaggle.json").exists():
        print("Upload kaggle.json")
        uploaded = files.upload()
        Path("/root/.kaggle").mkdir(parents=True, exist_ok=True)
        if "kaggle.json" in uploaded:
            !cp kaggle.json /root/.kaggle/kaggle.json
            !chmod 600 /root/.kaggle/kaggle.json
else:
    print("Pastikan ~/.kaggle/kaggle.json tersedia bila memakai lokal/server.")

## 2. Download dataset utama
Dataset utama: `mohamedhanyyy/chest-ctscan-images`.

Mirror duplikat `kabil007/lungcancer4types-imagedataset` tidak dipakai secara default.

In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/lungscan_data")
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_DATASET = "mohamedhanyyy/chest-ctscan-images"
SECOND_MIRROR_DATASET = "kabil007/lungcancer4types-imagedataset"
USE_SECOND_MIRROR = False

datasets = [PRIMARY_DATASET]
if USE_SECOND_MIRROR:
    datasets.append(SECOND_MIRROR_DATASET)

for slug in datasets:
    out_dir = RAW_DIR / slug.replace("/", "__")
    out_dir.mkdir(parents=True, exist_ok=True)
    print("Downloading", slug)
    !kaggle datasets download -d {slug} -p {out_dir} --unzip

## 3. Scan gambar dan mapping label
Notebook membaca semua `.jpg`, `.jpeg`, `.png`, `.bmp`, `.webp`, lalu mengambil label dari path/folder.

In [ ]:
import re, hashlib
import pandas as pd
from pathlib import Path

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
LABEL_MAP = {
    "adenocarcinoma": 0,
    "large_cell_carcinoma": 1,
    "squamous_cell_carcinoma": 2,
    "normal": 3,
}
IDX_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}

def infer_label(path):
    s = str(path).lower()
    s_norm = s.replace("_", " ").replace("-", " ")
    if "adenocarcinoma" in s_norm:
        return "adenocarcinoma"
    if "large.cell" in s or "large cell" in s_norm or "largecell" in s_norm:
        return "large_cell_carcinoma"
    if "squamous" in s_norm:
        return "squamous_cell_carcinoma"
    if re.search(r"(^|/|\\|\s)normal($|/|\\|\s)", s_norm):
        return "normal"
    return None

def infer_split(path):
    parts = [p.lower() for p in Path(path).parts]
    if any(p in {"train", "training"} for p in parts): return "train"
    if any(p in {"valid", "validation", "val"} for p in parts): return "valid"
    if any(p in {"test", "testing"} for p in parts): return "test"
    return "unknown"

rows=[]
for p in RAW_DIR.rglob("*"):
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
        label = infer_label(p)
        if label:
            rows.append({"path": str(p), "label": label, "label_id": LABEL_MAP[label], "split": infer_split(p)})

df = pd.DataFrame(rows)
print("Jumlah gambar:", len(df))
display(df.head())
display(pd.crosstab(df["split"], df["label"]))

## 4. Validasi gambar dan hapus duplikat
Jika mirror dataset ikut dimasukkan, cell ini akan menghapus duplikat exact-copy memakai SHA256.

In [ ]:
from PIL import Image
from tqdm.auto import tqdm

if len(df) == 0:
    raise ValueError("Tidak ada gambar terbaca. Cek download dan struktur folder.")

def valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False

df["valid"] = [valid_image(p) for p in tqdm(df["path"], desc="Validasi gambar")]
print("File rusak:", (~df["valid"]).sum())
df = df[df["valid"]].copy()

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

df["sha256"] = [sha256_file(p) for p in tqdm(df["path"], desc="Hashing")]
print("Baris duplikat:", df.duplicated("sha256", keep=False).sum())

priority = {"test": 0, "valid": 1, "train": 2, "unknown": 3}
df["priority"] = df["split"].map(priority).fillna(3)
before = len(df)
df = df.sort_values(["sha256", "priority"]).drop_duplicates("sha256", keep="first").drop(columns=["priority"]).reset_index(drop=True)
print(f"Setelah dedupe: {len(df)} gambar, terhapus {before-len(df)}")
display(pd.crosstab(df["split"], df["label"]))

## 5. Split train/valid/test
Jika folder split sudah lengkap, gunakan split bawaan dataset. Jika tidak, buat split stratified baru.

In [ ]:
from sklearn.model_selection import train_test_split

required = {"train", "valid", "test"}
if not required.issubset(set(df["split"].unique())) or "unknown" in set(df["split"].unique()):
    train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label_id"], random_state=42)
    valid_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label_id"], random_state=42)
    train_df = train_df.copy(); train_df["split"] = "train"
    valid_df = valid_df.copy(); valid_df["split"] = "valid"
    test_df = test_df.copy(); test_df["split"] = "test"
else:
    train_df = df[df["split"] == "train"].copy()
    valid_df = df[df["split"] == "valid"].copy()
    test_df = df[df["split"] == "test"].copy()

print(len(train_df), len(valid_df), len(test_df))
display(train_df["label"].value_counts())
display(valid_df["label"].value_counts())
display(test_df["label"].value_counts())

## 6. TensorFlow dataset pipeline

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
NUM_CLASSES = 4
AUTOTUNE = tf.data.AUTOTUNE
SEED = 42

def make_dataset(dataframe, shuffle=False):
    paths = dataframe["path"].values
    labels = dataframe["label_id"].values.astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def load_image(path, label):
        img = tf.io.read_file(path)
        img = tf.io.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.float32)
        return img, label

    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(min(len(dataframe), 1000), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, True)
valid_ds = make_dataset(valid_df, False)
test_ds = make_dataset(test_df, False)

for x, y in train_ds.take(1):
    print(x.shape, y.shape, tf.reduce_min(x).numpy(), tf.reduce_max(x).numpy())

## 7. Preview sample

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,10))
for images, labels in train_ds.take(1):
    for i in range(min(9, images.shape[0])):
        plt.subplot(3,3,i+1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(IDX_TO_LABEL[int(labels[i])])
        plt.axis("off")
plt.tight_layout()
plt.show()

## 8. Class weight

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
classes = np.array(sorted(LABEL_MAP.values()))
weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_df["label_id"].values)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
class_weight

## 9. Build model EfficientNetB0 baseline

In [ ]:
base_model = keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
base_model.trainable = False

augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.04),
    layers.RandomZoom(0.08),
    layers.RandomContrast(0.10),
], name="augmentation")

inputs = keras.Input(shape=IMG_SIZE + (3,), name="ct_image")
x = augmentation(inputs)
x = keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = layers.Dropout(0.35, name="dropout")(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="class_probability")(x)

model = keras.Model(inputs, outputs, name="lungscan_ai_effnetb0_baseline")
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 10. Training tahap 1

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint("best_lungscan_model.keras", monitor="val_accuracy", save_best_only=True, mode="max", verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=1),
]

history_head = model.fit(train_ds, validation_data=valid_ds, epochs=8, class_weight=class_weight, callbacks=callbacks)

## 11. Fine-tuning tahap 2

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_fine = model.fit(train_ds, validation_data=valid_ds, epochs=12, class_weight=class_weight, callbacks=callbacks)

## 12. Plot training history

In [ ]:
def plot_history(histories):
    acc, val_acc, loss, val_loss = [], [], [], []
    for h in histories:
        acc += h.history.get("accuracy", [])
        val_acc += h.history.get("val_accuracy", [])
        loss += h.history.get("loss", [])
        val_loss += h.history.get("val_loss", [])
    epochs = range(1, len(acc)+1)

    plt.figure(figsize=(8,5))
    plt.plot(epochs, acc, label="train accuracy")
    plt.plot(epochs, val_acc, label="valid accuracy")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("Accuracy")
    plt.legend(); plt.grid(True, alpha=0.25); plt.show()

    plt.figure(figsize=(8,5))
    plt.plot(epochs, loss, label="train loss")
    plt.plot(epochs, val_loss, label="valid loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Loss")
    plt.legend(); plt.grid(True, alpha=0.25); plt.show()

plot_history([history_head, history_fine])

## 13. Evaluasi test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

best_model = keras.models.load_model("best_lungscan_model.keras")
test_loss, test_acc = best_model.evaluate(test_ds)
print("Test accuracy:", test_acc)

y_true, y_pred = [], []
for images, labels in test_ds:
    probs = best_model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(probs, axis=1).tolist())

names = [IDX_TO_LABEL[i] for i in range(NUM_CLASSES)]
print(classification_report(y_true, y_pred, target_names=names, digits=4))
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(9,8))
ConfusionMatrixDisplay(cm, display_labels=names).plot(ax=ax, xticks_rotation=45, values_format="d")
plt.title("Confusion Matrix")
plt.tight_layout(); plt.show()

## 14. Simulasi response API `/api/predict`

In [ ]:
def predict_single_image(model, image_path):
    img = tf.io.read_file(str(image_path))
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    probs = model.predict(tf.expand_dims(img, 0), verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    return {
        "prediction": IDX_TO_LABEL[pred_idx],
        "confidence": round(float(probs[pred_idx] * 100), 2),
        "all_scores": {IDX_TO_LABEL[i]: round(float(probs[i] * 100), 2) for i in range(NUM_CLASSES)},
        "model_accuracy": round(float(test_acc * 100), 2),
    }

sample_path = test_df.iloc[0]["path"]
api_response = predict_single_image(best_model, sample_path)
api_response

## 15. Grad-CAM dasar
Untuk produksi, backend FastAPI dapat mengubah heatmap menjadi `heatmap_base64`.

In [ ]:
# Ambil base model dari best_model yang sudah diload, lalu cari Conv2D terakhir.
base_for_gradcam = best_model.get_layer("efficientnetb0")

last_conv_layer_name = None
for layer in reversed(base_for_gradcam.layers):
    if isinstance(layer, layers.Conv2D):
        last_conv_layer_name = layer.name
        break
print("Last conv layer:", last_conv_layer_name)

def make_gradcam_heatmap(img_array, model, base_model_layer, layer_name, pred_index=None):
    # Grad model memakai input base EfficientNet, bukan input outer model, agar tidak graph-disconnected.
    grad_base = keras.models.Model(
        base_model_layer.inputs,
        [base_model_layer.get_layer(layer_name).output, base_model_layer.output]
    )

    # Untuk Grad-CAM, jangan pakai augmentation acak. Preprocess sama seperti inferensi.
    preprocessed = keras.applications.efficientnet.preprocess_input(img_array)

    with tf.GradientTape() as tape:
        conv_outputs, base_output = grad_base(preprocessed)
        x = model.get_layer("global_average_pooling")(base_output)
        x = model.get_layer("dropout")(x, training=False)
        predictions = model.get_layer("class_probability")(x)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.squeeze(conv_outputs[0] @ pooled_grads[..., tf.newaxis])
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def load_img(path):
    img = tf.io.read_file(str(path))
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return tf.cast(img, tf.float32)

sample_img = load_img(sample_path)
heatmap = make_gradcam_heatmap(tf.expand_dims(sample_img, 0), best_model, base_for_gradcam, last_conv_layer_name)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1); plt.imshow(sample_img.numpy().astype("uint8")); plt.title("CT-scan"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(sample_img.numpy().astype("uint8")); plt.imshow(heatmap, cmap="jet", alpha=0.45); plt.title("Grad-CAM"); plt.axis("off")
plt.tight_layout(); plt.show()

## 16. Simpan model `.keras`, `.h5`, dan label mapping

In [ ]:
import json
EXPORT_DIR = Path("/content/lungscan_exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

best_model.save(EXPORT_DIR / "lungscan_ai_baseline.keras")
best_model.save(EXPORT_DIR / "lungscan_ai_baseline.h5")

with open(EXPORT_DIR / "label_mapping.json", "w") as f:
    json.dump({
        "label_to_idx": LABEL_MAP,
        "idx_to_label": {str(k): v for k, v in IDX_TO_LABEL.items()},
        "img_size": list(IMG_SIZE),
        "model_name": "lungscan_ai_effnetb0_baseline"
    }, f, indent=2)

print("Export:", EXPORT_DIR)

## 17. Download artifact dari Colab

In [ ]:
try:
    from google.colab import files
    files.download(str(EXPORT_DIR / "lungscan_ai_baseline.h5"))
    files.download(str(EXPORT_DIR / "label_mapping.json"))
except Exception:
    print("Download manual dari:", EXPORT_DIR)

## 18. Next step

Setelah baseline berhasil:
1. Cek confusion matrix, terutama kanker yang salah menjadi `normal`.
2. Jangan percaya akurasi tinggi jika dataset kecil/mirror.
3. Tambahkan dataset klinis yang lebih kuat untuk validasi tahap berikutnya.
4. Pastikan preprocessing di FastAPI sama persis dengan notebook ini.
5. Selalu tampilkan disclaimer medis pada website dan laporan PDF.